# Course 10 — Governed durable state and memory

**Scenario:** Northstar's EU checkout fails after `deploy-1842`. The agent may read health, logs, and deployment evidence, form a hypothesis, and prepare a rollback proposal. It never executes rollback in this core lab.

> Persistence changes the safety model. State can outlive the process, deployment, policy, credentials, and human who started the run. Resume, replay, forks, and memory therefore require explicit governance.

## Learning path and safety boundaries

This notebook uses one shared Pydantic implementation from `policy.py` and `lab.py`. It proceeds through typed state, durable checkpoints, a real two-process restart, replay safety, persistent budgets, structured approval, immutable forks, version checks, governed memory, safe streaming, measured evaluation, and an optional current LangGraph adapter.

The default path is deterministic, credential-free, and synthetic. It stores compact evidence metadata—not raw logs, credentials, hidden reasoning, or production secrets.

## Setup

Pydantic and the Python standard library are sufficient for the core lab. LangGraph is optional and imported only when installed. A temporary directory keeps every SQLite artifact local to this notebook run.

In [ ]:
from datetime import timedelta
from importlib.util import find_spec
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

candidates = [Path.cwd(), Path.cwd() / 'curriculum/intermediate/10-langgraph-state-memory']
COURSE_DIR = next(path for path in candidates if (path / 'policy.py').exists())
sys.path.insert(0, str(COURSE_DIR))

import lab
from policy import (
    ExecutionMode, GovernedMemoryStore, MemoryType, PolicyError,
    SQLiteCheckpointRepository, TerminalStatus,
)

temporary_directory = TemporaryDirectory()
workspace = Path(temporary_directory.name)
context = lab.build_context()
print(context.model_dump())

## Part 1 — Typed thread and incident-state contracts

`ThreadContext` is application-owned identity and authority. `IncidentState` persists the request, compact evidence, hypothesis, node attempts, tool and recovery budgets, structured pending approval, terminal status, versions, and latest checkpoint ID. Frozen records and `extra='forbid'` prevent silent field injection.

In [ ]:
repository = SQLiteCheckpointRepository(workspace / 'typed-state.sqlite')
runtime = lab.IncidentRuntime(repository, context)
initial = runtime.start(tool_budget=5)
assert initial.remaining_budget == 5
assert initial.evidence == ()
assert 'api_token' not in initial.model_dump()
print(initial.model_dump(exclude={'evidence', 'pending_approval'}))

## Part 2 — Durable checkpoint repository

The repository implements `save`, `load`, `history`, `fork`, and deletion over SQLite using typed JSON and state digests. Every read validates thread, tenant, owner, authorization scope, retention, policy, and optional schema/graph compatibility. The thread ID locates state; it does not authorize access.

In [ ]:
history = repository.history(context, now=lab.FIXED_TIME)
assert len(history) == 1
attacker = lab.build_context(tenant_id='globex')
denial_code = None
try:
    repository.load(attacker, now=lab.FIXED_TIME)
except PolicyError as error:
    denial_code = error.code
assert denial_code == 'THREAD_ACCESS_DENIED'
print(denial_code)

## Part 3 — Northstar investigation

Each read produces a compact `EvidenceRecord`: stable ID, source/version, observation time, summary, artifact handle, hash, and correlation group. Raw logs remain outside graph state. Independent log and deployment groups support the hypothesis; repeated copies do not.

In [ ]:
runtime.read_health()
runtime.read_logs()
runtime.read_deployment()
state = runtime.form_hypothesis()
assert state.hypothesis == 'deploy-1842 introduced payment configuration v42.'
assert state.confidence == 1.0
assert state.tool_call_count == 3 and state.remaining_budget == 2
[(item.evidence_id, item.artifact_handle) for item in state.evidence]

## Part 4 — Real failure and process reconstruction

This is not a second object sharing an in-memory saver. Phase one and phase two run in different OS processes against the same durable database. The second process loads the authorized checkpoint and skips completed health/log reads.

In [ ]:
restart = lab.run_process_restart_experiment(workspace / 'process-restart.sqlite')
assert restart.phase_one_pid != restart.phase_two_pid
assert restart.calls_before_restart == 2
assert restart.calls_after_resume == 3
assert restart.remaining_before_restart == 3
assert restart.remaining_after_resume == 2
assert restart.repeated_completed_nodes == ()
restart.model_dump()

## Part 5 — Replay-safe reducers and confidence

Reducers merge evidence by stable ID and reject ID/hash conflicts. Confidence counts independent correlation groups, not list length. Separately, the replay experiment shows why a side effect before an interrupt can commit twice and why a stable logical operation ID plus unique attempt IDs produces one mock receipt.

In [ ]:
one = lab.merge_evidence((), (lab.logs_evidence(),))
replayed = lab.merge_evidence(one, (lab.logs_evidence(),))
assert len(replayed) == 1
assert lab.confidence_from_independent_evidence(one, 'deploy-1842') == 0.5
assert lab.confidence_from_independent_evidence(replayed, 'deploy-1842') == 0.5
replay = lab.run_replay_experiment()
assert replay.bad_external_commit_count == 2
assert replay.safe_external_commit_count == 1
replay.model_dump()

## Part 6 — Budgets and retries survive resume

A restart must not mint new work. Tool calls, remaining budget, deadline, per-node attempts, retry allowance, and replan/reflection allowances are persisted together. A timeout before restart and retry afterward belong to one retry budget.

In [ ]:
retry_repo = SQLiteCheckpointRepository(workspace / 'retry.sqlite')
retry_runtime = lab.IncidentRuntime(retry_repo, context)
retry_runtime.start()
retry_runtime.read_logs(timeout=True)
retry_runtime = lab.IncidentRuntime(retry_repo, context)
retry_runtime.resume_existing()
retry_runtime.read_logs(timeout=True)
attempts = {item.node: item.attempts for item in retry_runtime.state.attempts_by_node}
assert attempts['read-logs'] == 2
assert retry_runtime.state.retry_budget_remaining == 0
retry_runtime.state.model_dump(include={'tool_call_count', 'remaining_budget', 'retry_budget_remaining'})

## Part 7 — Structured approval interrupt

Approval is not a mutable boolean. The proposal binds tenant, action, target, evidence, policy, expiry, creator, and logical operation digest. The decision also binds the authenticated approver. The core lab checkpoints a safe review payload and stops at `INTERRUPTED`; it performs no rollback.

In [ ]:
approval_repo = SQLiteCheckpointRepository(workspace / 'approval.sqlite')
approval_runtime = lab.IncidentRuntime(approval_repo, context)
paused = approval_runtime.run_to_interrupt()
proposal = paused.pending_approval
assert paused.terminal_status == TerminalStatus.INTERRUPTED
assert proposal.target == 'deploy-1842'
assert proposal.digest
proposal.model_dump(exclude={'evidence_ids'})

## Part 8 — Stale, changed, and cancelled approvals

An approval for `deploy-1842` cannot authorize `deploy-1843`. Expired decisions fail. Cancellation is terminal, so a late approval cannot revive the run. A valid decision completes only the approval handoff—the external action remains outside this course.

In [ ]:
old_decision = lab.build_decision(proposal)
revised = approval_runtime.revise_proposal_target('deploy-1843')
assert revised.digest != proposal.digest
stale_code = None
try:
    approval_runtime.resume_with_approval(old_decision, lab.build_approver(), now=lab.FIXED_TIME + timedelta(minutes=10))
except PolicyError as error:
    stale_code = error.code
assert stale_code == 'APPROVAL_TARGET_MISMATCH'

def new_paused_runtime(name):
    repo = SQLiteCheckpointRepository(workspace / f'approval-{name}.sqlite')
    runtime = lab.IncidentRuntime(repo, context)
    return repo, runtime, runtime.run_to_interrupt().pending_approval

expired_repo, expired_runtime, expired_proposal = new_paused_runtime('expired')
expired_code = None
try:
    expired_runtime.resume_with_approval(lab.build_decision(expired_proposal), lab.build_approver(), now=expired_proposal.expires_at + timedelta(seconds=1))
except PolicyError as error:
    expired_code = error.code
assert expired_code == 'APPROVAL_EXPIRED'

cancel_repo, cancel_runtime, cancel_proposal = new_paused_runtime('cancelled')
late_decision = lab.build_decision(cancel_proposal)
cancel_runtime.cancel()
cancelled_code = None
try:
    cancel_runtime.resume_with_approval(late_decision, lab.build_approver(), now=lab.FIXED_TIME + timedelta(minutes=10))
except PolicyError as error:
    cancelled_code = error.code
assert cancelled_code == 'CANCELLED_RUN'

valid_repo, valid_runtime, current_proposal = new_paused_runtime('valid')
approved = valid_runtime.resume_with_approval(lab.build_decision(current_proposal), lab.build_approver(), now=lab.FIXED_TIME + timedelta(minutes=10))
assert approved.terminal_status == TerminalStatus.COMPLETED
assert approved.pending_approval is None
print({'stale': stale_code, 'expired': expired_code, 'cancelled': cancelled_code})

## Part 9 — Immutable history and safe time-travel forks

Time travel never mutates a historical checkpoint. `fork()` creates lineage through `parent_checkpoint_id`, clears approval, assigns a new run, and defaults to `REPLAY`. The mock executor refuses external commits outside `LIVE` mode.

In [ ]:
source_checkpoint = approval_repo.history(context, now=lab.FIXED_TIME)[1]
fork_context = lab.build_context(thread_id='thread-northstar-eu-fork')
fork = approval_repo.fork(context, source_checkpoint.checkpoint_id, fork_context, now=lab.FIXED_TIME)
assert fork.record.parent_checkpoint_id == source_checkpoint.checkpoint_id
assert fork.state.execution_mode == ExecutionMode.REPLAY
executor = lab.IdempotentMockExecutor()
receipt = executor.execute(fork.state.logical_operation_id, 'attempt-fork-1', mode=fork.state.execution_mode)
assert receipt.status.value == 'DRY_RUN' and executor.external_commit_count == 0
fork.record.model_dump(include={'checkpoint_id', 'parent_checkpoint_id', 'execution_mode'})

## Part 10 — Schema, graph, and policy versions

A state-v1 checkpoint must not silently resume under a state-v2 runtime, and graph-v3 execution state may be incompatible with graph-v5 scheduling. Resume also revalidates current policy. The fixture returns explicit `MIGRATION_REQUIRED`, `GRAPH_VERSION_MISMATCH`, or `POLICY_REVALIDATION_REQUIRED` outcomes.

In [ ]:
version_codes = []
for expected_schema, expected_graph in [('state-v3', None), (None, 'graph-v5')]:
    try:
        approval_repo.load(context, expected_state_schema_version=expected_schema, expected_graph_version=expected_graph, now=lab.FIXED_TIME)
    except PolicyError as version_error:
        version_codes.append(version_error.code)
assert version_codes == ['MIGRATION_REQUIRED', 'GRAPH_VERSION_MISMATCH']
print(version_codes)

## Part 11 — Long-term memory write policy

Persistence is not memory, and memory is not evidence. `GovernedMemoryStore` uses tenant + subject + memory type namespaces. It admits user-confirmed preferences, reviewed postmortems, and approved procedures; it denies unverified hypotheses, retrieved instructions, authorization claims, and credentials. Storage could be KV, relational, document, vector, or hybrid—the policy semantics stay the same.

In [ ]:
memory_store = GovernedMemoryStore()
preference_v1, preference_v2 = lab.seed_memory_store(memory_store, context)
denial_code = None
try:
    memory_store.write(context, lab.memory_poisoning_request(), now=lab.FIXED_TIME)
except PolicyError as memory_error:
    denial_code = memory_error.code
assert denial_code == 'MEMORY_WRITE_DENIED'
print(denial_code)

## Part 12 — Verified memory versus a relevant hunch

The unverified memory “Checkout incidents are usually Redis” scores 0.99 relevance, but relevance is not verification. Retrieval excludes it. The current root-cause claim remains grounded only in this run's deployment and log evidence.

In [ ]:
memories = memory_store.retrieve(context, subject_id='checkout-eu', now=lab.FIXED_TIME)
memory_ids = {item.memory_id for item in memories}
assert 'memory-redis-hunch' not in memory_ids
projected_context = lab.incident_context(state, memories)
assert projected_context['memory_used_as_incident_evidence'] is False
[(item.memory_type.value, item.content) for item in memories]

## Part 13 — Tenant, expiry, supersession, and deletion

Normal retrieval excludes Globex data, expired records, superseded preference v1, and deleted records. The active preference is v2 (“concise updates”). Records under required audit retention cannot be casually deleted.

In [ ]:
preferences = memory_store.retrieve(context, subject_id='checkout-eu', memory_types=(MemoryType.PREFERENCE,), now=lab.FIXED_TIME)
assert [item.version for item in preferences] == [2]
assert all(item.tenant_id == 'northstar' for item in preferences)
memory_store.delete(context, preference_v2.memory_id, now=lab.FIXED_TIME)
after_delete = memory_store.retrieve(context, subject_id='checkout-eu', now=lab.FIXED_TIME)
assert preference_v2.memory_id not in {item.memory_id for item in after_delete}
print('governed memory lifecycle passed')

## Part 14 — Correlated events and safe streaming

Internal events retain run, thread, checkpoint, sequence, timestamp, and type for audit correlation. `project_stream_event()` exposes only node, status, elapsed milliseconds, and a safe summary. Raw logs, PII, credentials, and hidden reasoning never enter the UI projection.

In [ ]:
safe_events = runtime.safe_stream()
rendered = str([event.model_dump() for event in safe_events])
assert 'secret-token' not in rendered
assert 'customer@example.com' not in rendered
assert set(safe_events[0].model_dump()) == {'node', 'status', 'elapsed_ms', 'safe_summary'}
safe_events[-3:]

## Part 15 — Evaluation and persistence baseline

The evaluation derives rates from observable attempts/outcomes and measures checkpoint latency/size from actual saves. The baseline comparison uses observed restart counts: without persistence, a fresh run repeats health and logs; durable resume performs only the missing deployment read.

In [ ]:
metrics = lab.run_evaluation(workspace / 'evaluation.sqlite')
no_persistence, durable = lab.compare_persistence(restart)
assert metrics.resume_success_rate == 1.0
assert metrics.duplicate_work_rate == 0.0
assert metrics.memory_contamination_rate == 0.0
assert durable.tool_calls < no_persistence.tool_calls
print(metrics.model_dump())
print([no_persistence.model_dump(), durable.model_dump()])

## Part 16 — Optional current LangGraph adapter

The core contracts do not require LangGraph. When the optional package is installed, `build_optional_langgraph_adapter()` uses current `StateGraph`, `InMemorySaver`, `InMemoryStore`, `interrupt(...)`, and `Command(resume=...)` APIs. `InMemorySaver` is development-only; SQLite and PostgreSQL savers are separate packages. LangGraph may re-run code before an interrupt, so authorization and idempotency remain outside the framework.

In [ ]:
if find_spec('langgraph'):
    from langgraph.types import Command
    graph = lab.build_optional_langgraph_adapter()
    config = {'configurable': {'thread_id': 'course-10-optional'}}
    first = graph.invoke({'request': 'Prepare rollback', 'safe_review_payload': {}, 'approval_decision': None}, config=config)
    assert first['__interrupt__']
    resumed = graph.invoke(Command(resume={'decision': 'APPROVE'}), config=config)
    assert resumed['approval_decision'] == {'decision': 'APPROVE'}
    print('optional LangGraph interrupt/resume passed')
else:
    print('Optional LangGraph extra not installed; deterministic core completed.')

## Production upgrade path

| Teaching fixture | Production control |
| --- | --- |
| temporary SQLite repository | supported durable checkpointer, migrations, pooling, backup/restore |
| typed JSON + digest | encrypted serialization, key management, integrity monitoring |
| compact synthetic evidence | governed artifact service with provenance and retention |
| deterministic memory store | durable tenant/subject/type namespaces with deletion and audit workflows |
| mock receipt | Course 03 approval + transactional outbox/idempotent executor |
| local safe projections | authenticated event transport, redaction tests, access-controlled telemetry |

Reacquire credentials at resume time. Never persist API keys, database passwords, temporary cloud credentials, bearer approval tokens, raw logs, PII, or hidden reasoning.

## Exercises

1. Write a state-v1 → state-v2 migration that preserves the original checkpoint.
2. Inject the same evidence ID with a different hash and explain the fail-closed behavior.
3. Add an audit-retained procedural memory and prove that it informs a checklist but cannot authorize rollback.
4. Replace the optional in-memory adapter with `SqliteSaver` and document package, setup, and concurrency limits.
5. Add a conflicting independent source and inspect `NEEDS_HUMAN_REVIEW` plus the redacted stream.

## Summary

Persistence is not memory. Memory is not evidence. Thread ID is not authority. Interrupt is not authorization. Resume is not a fresh run. Replay can repeat code. Historical state is immutable and time travel is a safe fork. Versions, policy, retention, identity, credentials, and approval must be checked again whenever durable state returns to life.